In [1]:
import pandas as pd
import numpy as np
import sys
import os

In [2]:
# Load the historical flights dataset
print("Loading historical flights dataset...")
data_path = "/home/kristsey/AvinorHackathon/data/avinor_data/historical_flights.csv"

# Load data
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

Loading historical flights dataset...
Dataset shape: (410442, 11)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410442 entries, 0 to 410441
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   flight_id          410442 non-null  object
 1   dep_airport        410442 non-null  object
 2   dep_airport_group  232968 non-null  object
 3   arr_airport        410442 non-null  object
 4   arr_airport_group  231669 non-null  object
 5   service_type       410336 non-null  object
 6   std                410442 non-null  object
 7   sta                410442 non-null  object
 8   cancelled          410442 non-null  int64 
 9   atd                394327 non-null  object
 10  ata                393576 non-null  object
dtypes: int64(1), object(10)
memory usage: 34.4+ MB
None

First few rows:
  flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
0     WF149         HOV            

In [3]:
# Load the october schedule dataset
print("Testing for unscheduled service types")
scheduled_data_path = (
    "/home/kristsey/AvinorHackathon/data/avinor_data/schedule_oct2025.csv"
)

# Load data
scheduled_data_october = pd.read_csv(scheduled_data_path)


print(f"Dataset shape: {df.shape}")
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.isna().sum())

historical_flight_service_types = df["service_type"].unique()
scheduled_flight_service_types = scheduled_data_october["service_type"].unique()
print("Historical flight service types:", historical_flight_service_types)
print("Scheduled flight service types:", scheduled_flight_service_types)
unscheduled_service_types = set(scheduled_flight_service_types) - set(
    historical_flight_service_types
)
print("Unscheduled service types in scheduled data:", unscheduled_service_types)
# Remove rows with missing actual departure or arrival times

Testing for unscheduled service types
Dataset shape: (410442, 11)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410442 entries, 0 to 410441
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   flight_id          410442 non-null  object
 1   dep_airport        410442 non-null  object
 2   dep_airport_group  232968 non-null  object
 3   arr_airport        410442 non-null  object
 4   arr_airport_group  231669 non-null  object
 5   service_type       410336 non-null  object
 6   std                410442 non-null  object
 7   sta                410442 non-null  object
 8   cancelled          410442 non-null  int64 
 9   atd                394327 non-null  object
 10  ata                393576 non-null  object
dtypes: int64(1), object(10)
memory usage: 34.4+ MB
None

First few rows:
flight_id                 0
dep_airport               0
dep_airport_group    177474
arr_airport               

In [4]:
# Add the parent directory to the Python path so we can import from src
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath("avinorhackathon")))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print(sys.path)

# Now we can import from src
from src.preprocessing import DataPreprocessor

data_preprocessor = DataPreprocessor()
df_processed = data_preprocessor.process_hist_flights(df)

na_airport_group_data = df_processed[
    df_processed["arr_airport_group"].isna() | df_processed["dep_airport_group"].isna()
]

print(na_airport_group_data.head())


# Remove rows with missing actual departure or arrival times
df_schedule = df_processed.drop(["atd", "ata"], axis=1).dropna()

print(f"Dataset shape after removing actual values", df_schedule.isna().sum())

['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/lhome/kristsey/AvinorHackathon/.venv/lib/python3.12/site-packages', '/lhome/kristsey/AvinorHackathon']
Missing values after processing:
flight_id                 0
dep_airport               0
dep_airport_group    169367
arr_airport               0
arr_airport_group    169919
service_type              0
std                       0
sta                       0
atd                       0
ata                       0
dtype: int64
    flight_id dep_airport dep_airport_group arr_airport arr_airport_group  \
362    WF7101         SKN               NaN         MJF                 D   
363    WF8102         MJF                 D         BOO               NaN   
364     WF743         MJF                 D         OSL               NaN   
365    WF7025         OSL               NaN         MJF                 D   
366    SK4516         OSL               NaN         MOL                 B   

    service_type 

In [26]:
def create_concurrency_binary_target(df):
    """
    Create a binary target indicating whether concurrency > 0
    """
    df = df.copy()
    df["is_concurrency"] = (df["concurrency"] > 0).astype(int)
    return df

In [ ]:
def compute_scheduled_concurrency(df, include_features=False):
    """ "
    For every airport group, and hour from first scheduled time to last scheduled time, count the concurrency

    Concurrency is defined as the number of times an airport group has an overlapping scheduled departure or arrival within a certain timeframe

    The timeframe is defined as 16 minutes before to 5 minutes after a landing, or 15 minutes before to 8 inutes after takeoff
    """
    scheduled_df = df.copy()
    concurrency_records = []

    # Convert scheduled times to datetime
    scheduled_df["std"] = pd.to_datetime(scheduled_df["std"])
    scheduled_df["sta"] = pd.to_datetime(scheduled_df["sta"])

    # Get the range of hours to consider
    min_time = min(scheduled_df["std"].min(), scheduled_df["sta"].min()).floor("H")
    max_time = max(scheduled_df["std"].max(), scheduled_df["sta"].max()).ceil("H")

    # Create a time range with hourly frequency
    time_range = pd.date_range(start=min_time, end=max_time, freq="H")

    # Iterate over each airport group
    airport_groups = (
        pd.concat(
            [scheduled_df["dep_airport_group"], scheduled_df["arr_airport_group"]]
        )
        .dropna()
        .unique()
    )

    dep_scheduled_df = scheduled_df[scheduled_df["dep_airport_group"].notna().copy()]
    arr_scheduled_df = scheduled_df[scheduled_df["arr_airport_group"].notna().copy()]

    dep_scheduled_df["time_window_start"] = dep_scheduled_df["std"] - pd.Timedelta(
        minutes=15
    )
    dep_scheduled_df["time_window_end"] = dep_scheduled_df["std"] + pd.Timedelta(
        minutes=8
    )

    arr_scheduled_df["time_window_start"] = arr_scheduled_df["sta"] - pd.Timedelta(
        minutes=16
    )
    arr_scheduled_df["time_window_end"] = arr_scheduled_df["sta"] + pd.Timedelta(
        minutes=5
    )

    dep_scheduled_df = dep_scheduled_df.rename(
        columns={"dep_airport_group": "airport_group"}
    ).drop(columns=["arr_airport_group"])
    arr_scheduled_df = arr_scheduled_df.rename(
        columns={"arr_airport_group": "airport_group"}
    ).drop(columns=["dep_airport_group"])

    dep_arr_scheduled_df = pd.concat([dep_scheduled_df, arr_scheduled_df])

    concurrency_df = pd.DataFrame(
        columns=["airport_group", "date", "hour", "concurrency"]
    )
    concurrency_records = []

    # start_time = dep_arr_scheduled_df["std"].min().floor("m")
    # end_time = dep_arr_scheduled_df["sta"].max().ceil("m")
    start_time = (
        dep_arr_scheduled_df["std"].min().floor("D").replace(day=1)
    )  # Start of month
    end_time = (
        dep_arr_scheduled_df["sta"].max().ceil("D").replace(day=1)
        + pd.DateOffset(months=1)
        - pd.Timedelta(days=1)
    ).ceil(
        "D"
    )  # End of month

    time_index = pd.date_range(start=start_time, end=end_time, freq="h")

    for airport_group in airport_groups:
        print(f"Processing airport group: {airport_group}")
        # iterate over time index
        for current_time in time_index:
            # Define the current hour window
            next_hour = current_time + pd.Timedelta(hours=1)

            # Get all flights for this airport group
            group_flights = dep_arr_scheduled_df[
                dep_arr_scheduled_df["airport_group"] == airport_group
            ]

            # Find flights whose time windows overlap with the current hour
            # A time window overlaps with the current hour if:
            # - The window starts before the hour ends AND
            # - The window ends after the hour starts
            flights_in_window = group_flights[
                (group_flights["time_window_start"] < next_hour)
                & (group_flights["time_window_end"] > current_time)
            ]

            # Now calculate the actual concurrency: count overlapping time windows between flights
            concurrency_count = 0
            concurrency_windows = []

            # For each flight in the window, count how many other flights overlap with it
            for i in range(len(flights_in_window)):
                flight_i = flights_in_window.iloc[i]
                for j in range(i + 1, len(flights_in_window)):
                    flight_j = flights_in_window.iloc[j]

                    # Check if time windows of flight_i and flight_j overlap
                    # Two time windows overlap if:
                    # - flight_i starts before flight_j ends AND
                    # - flight_j starts before flight_i ends
                    if (
                        flight_i["time_window_start"] < flight_j["time_window_end"]
                        and flight_j["time_window_start"] < flight_i["time_window_end"]
                    ):
                        concurrency_windows.append(
                            (
                                max(
                                    flight_i["time_window_start"],
                                    flight_j["time_window_start"],
                                ),
                                min(
                                    flight_i["time_window_end"],
                                    flight_j["time_window_end"],
                                ),
                            )
                        )
            # Merge overlapping concurrency windows
            if concurrency_windows:
                # Sort windows by start time
                concurrency_windows.sort(key=lambda x: x[0])
                merged_windows = [concurrency_windows[0]]

                for current_start, current_end in concurrency_windows[1:]:
                    last_start, last_end = merged_windows[-1]

                    if current_start <= last_end:  # Overlap
                        merged_windows[-1] = (last_start, max(last_end, current_end))
                    else:
                        merged_windows.append((current_start, current_end))

                concurrency_count = len(merged_windows)
            else:
                concurrency_count = 0

            overlapping_time = (
                sum((end - start).total_seconds() / 60 for start, end in merged_windows)
                if concurrency_count > 0
                else 0
            )

            if include_features:
                concurrency_records.append(
                    {
                        "airport_group": airport_group,
                        "date": current_time.date(),
                        "hour": current_time.hour,
                        "concurrency": concurrency_count,
                        "number_of_flights": len(flights_in_window),
                        "overlapping_time_minutes": overlapping_time,
                    }
                )
            else:
                concurrency_records.append(
                    {
                        "airport_group": airport_group,
                        "date": current_time.date(),
                        "hour": current_time.hour,
                        "concurrency": concurrency_count,
                    }
                )

    # Convert records to DataFrame
    concurrency_df = pd.DataFrame(concurrency_records)
    return concurrency_df

In [6]:
scheduled_concurrency_oct = compute_scheduled_concurrency(
    scheduled_data_october, include_features=True
)
scheduled_concurrency_oct = scheduled_concurrency_oct.sort_values(
    by=["date", "hour", "airport_group"]
).reset_index(drop=True)
print(scheduled_concurrency_oct.head())

with open("../data/concurrency/scheduled_concurrency_oct.csv", "w") as f:
    scheduled_concurrency_oct.to_csv(f, index=False)

print(
    "max overlap time (minutes):",
    scheduled_concurrency_oct["overlapping_time_minutes"].max(),
)

/tmp/ipykernel_1011790/3096129496.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  min_time = min(scheduled_df["std"].min(), scheduled_df["sta"].min()).floor("H")
/tmp/ipykernel_1011790/3096129496.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  max_time = max(scheduled_df["std"].max(), scheduled_df["sta"].max()).ceil("H")
/tmp/ipykernel_1011790/3096129496.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_range = pd.date_range(start=min_time, end=max_time, freq="H")
/tmp/ipykernel_1011790/3096129496.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dep_scheduled_df["ti

Processing airport group: G
Processing airport group: E
Processing airport group: E
Processing airport group: B
Processing airport group: B
Processing airport group: A
Processing airport group: A
Processing airport group: C
Processing airport group: C
Processing airport group: D
Processing airport group: D
Processing airport group: F
Processing airport group: F
  airport_group        date  hour  concurrency  number of flights  \
0             A  2025-10-01     0            0                  0   
1             B  2025-10-01     0            0                  0   
2             C  2025-10-01     0            0                  0   
3             D  2025-10-01     0            0                  0   
4             E  2025-10-01     0            0                  0   

   overlapping_time_minutes  
0                       0.0  
1                       0.0  
2                       0.0  
3                       0.0  
4                       0.0  
max overlap time (minutes): 61.0
  airpor

In [53]:
scheduled_concurrency = compute_scheduled_concurrency(df_schedule)
scheduled_concurrency = scheduled_concurrency.sort_values(
    by=["date", "hour"]
).reset_index(drop=True)
print(scheduled_concurrency.head())

with open("../data/concurrency/scheduled_concurrency.csv", "w") as f:
    scheduled_concurrency.to_csv(f, index=False)

/tmp/ipykernel_1005651/3409272634.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  min_time = min(scheduled_df["std"].min(), scheduled_df["sta"].min()).floor("H")
/tmp/ipykernel_1005651/3409272634.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  max_time = max(scheduled_df["std"].max(), scheduled_df["sta"].max()).ceil("H")
/tmp/ipykernel_1005651/3409272634.py:21: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_range = pd.date_range(start=min_time, end=max_time, freq="H")


Processing airport group: A


KeyboardInterrupt: 

In [27]:
def compute_scheduled_concurrency_optimized(df, include_features=False):
    """
    Optimized version of compute_scheduled_concurrency.
    Counts the number of distinct time periods where flight time windows overlap.
    """
    import numpy as np

    scheduled_df = df.copy()

    # Convert scheduled times to datetime
    scheduled_df["std"] = pd.to_datetime(scheduled_df["std"])
    scheduled_df["sta"] = pd.to_datetime(scheduled_df["sta"])

    # Prepare departure and arrival dataframes
    dep_df = scheduled_df[scheduled_df["dep_airport_group"].notna()].copy()
    arr_df = scheduled_df[scheduled_df["arr_airport_group"].notna()].copy()

    # Add time windows
    dep_df["time_window_start"] = dep_df["std"] - pd.Timedelta(minutes=15)
    dep_df["time_window_end"] = dep_df["std"] + pd.Timedelta(minutes=8)
    dep_df["airport_group"] = dep_df["dep_airport_group"]

    arr_df["time_window_start"] = arr_df["sta"] - pd.Timedelta(minutes=16)
    arr_df["time_window_end"] = arr_df["sta"] + pd.Timedelta(minutes=5)
    arr_df["airport_group"] = arr_df["arr_airport_group"]

    # Combine and sort by airport group and time window start
    combined_df = (
        pd.concat(
            [
                dep_df[["airport_group", "time_window_start", "time_window_end"]],
                arr_df[["airport_group", "time_window_start", "time_window_end"]],
            ]
        )
        .sort_values(["airport_group", "time_window_start"])
        .reset_index(drop=True)
    )

    # Create time range
    start_time = combined_df["time_window_start"].min().floor("D").replace(day=1)
    end_time = (
        combined_df["time_window_end"].max().ceil("D").replace(day=1)
        + pd.DateOffset(months=1)
        - pd.Timedelta(days=1)
    ).ceil("D")
    time_index = pd.date_range(start=start_time, end=end_time, freq="H")

    concurrency_records = []

    # Group by airport for efficient processing
    for airport_group, group_data in combined_df.groupby("airport_group"):
        print(f"Processing airport group: {airport_group}")

        # Convert to numpy arrays for faster operations
        starts = group_data["time_window_start"].values
        ends = group_data["time_window_end"].values

        # Process each hour for this airport group
        for current_time in time_index:
            next_hour = current_time + pd.Timedelta(hours=1)

            # Find flights whose time windows overlap with current hour using vectorized operations
            current_time_np = np.datetime64(current_time)
            next_hour_np = np.datetime64(next_hour)

            # Vectorized overlap check
            overlaps_hour = (starts < next_hour_np) & (ends > current_time_np)

            if not overlaps_hour.any():
                record = {
                    "airport_group": airport_group,
                    "date": current_time.date(),
                    "hour": current_time.hour,
                    "concurrency": 0,
                }
                if include_features:
                    record.update(
                        {
                            "number_of_flights": 0,
                            "overlapping_time_minutes": 0,
                        }
                    )
                concurrency_records.append(record)
                continue

            # Get overlapping flights
            overlapping_starts = starts[overlaps_hour]
            overlapping_ends = ends[overlaps_hour]
            num_flights = len(overlapping_starts)

            if num_flights <= 1:
                record = {
                    "airport_group": airport_group,
                    "date": current_time.date(),
                    "hour": current_time.hour,
                    "concurrency": 0,
                }
                if include_features:
                    record.update(
                        {
                            "number_of_flights": num_flights,
                            "overlapping_time_minutes": 0,
                        }
                    )
                concurrency_records.append(record)
                continue

            # Count merged overlap windows efficiently
            concurrency_count, overlapping_time = (
                count_merged_overlap_windows_optimized(
                    overlapping_starts, overlapping_ends, return_time=include_features
                )
            )

            record = {
                "airport_group": airport_group,
                "date": current_time.date(),
                "hour": current_time.hour,
                "concurrency": concurrency_count,
            }
            if include_features:
                record.update(
                    {
                        "number_of_flights": num_flights,
                        "overlapping_time_minutes": overlapping_time,
                    }
                )
            concurrency_records.append(record)

    return pd.DataFrame(concurrency_records)


def count_merged_overlap_windows_optimized(starts, ends, return_time=False):
    """
    Count the number of distinct time periods where overlaps occur.
    This matches the original algorithm's logic:
    1. Find all overlapping pairs and their intersection windows
    2. Merge overlapping/adjacent intersection windows
    3. Count the number of merged windows
    """
    if len(starts) <= 1:
        return (0, 0) if return_time else 0

    n = len(starts)
    overlap_windows = []

    # Find all overlapping pairs and their intersection windows
    for i in range(n):
        for j in range(i + 1, n):
            # Check if windows overlap
            if starts[i] < ends[j] and starts[j] < ends[i]:
                # Calculate intersection window
                overlap_start = max(starts[i], starts[j])
                overlap_end = min(ends[i], ends[j])
                overlap_windows.append((overlap_start, overlap_end))

    if not overlap_windows:
        return (0, 0) if return_time else 0

    # Sort windows by start time
    overlap_windows.sort(key=lambda x: x[0])

    # Merge overlapping/adjacent windows
    merged_windows = [overlap_windows[0]]

    for current_start, current_end in overlap_windows[1:]:
        last_start, last_end = merged_windows[-1]

        if current_start <= last_end:  # Overlap or adjacent
            merged_windows[-1] = (last_start, max(last_end, current_end))
        else:
            merged_windows.append((current_start, current_end))

    if return_time:
        # Calculate total overlapping time in minutes
        total_time = sum(
            (end - start).astype("timedelta64[m]").astype(float)
            for start, end in merged_windows
        )
        return len(merged_windows), total_time

    return len(merged_windows)

In [28]:
scheduled_concurrency_oct = compute_scheduled_concurrency_optimized(
    scheduled_data_october, include_features=True
)
scheduled_concurrency_oct = scheduled_concurrency_oct.sort_values(
    by=["date", "hour"]
).reset_index(drop=True)

scheduled_concurrency_oct = create_concurrency_binary_target(scheduled_concurrency_oct)

with open("../data/concurrency/scheduled_concurrency_oct_optimized.csv", "w") as f:
    scheduled_concurrency_oct.to_csv(f, index=False)

/tmp/ipykernel_1011790/2221210852.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_index = pd.date_range(start=start_time, end=end_time, freq="H")


Processing airport group: A
Processing airport group: B
Processing airport group: C
Processing airport group: D
Processing airport group: E
Processing airport group: F
Processing airport group: G


In [32]:
scheduled_concurrency = compute_scheduled_concurrency_optimized(
    df_schedule, include_features=True
)
scheduled_concurrency = scheduled_concurrency.sort_values(
    by=["date", "hour"]
).reset_index(drop=True)

scheduled_concurrency = create_concurrency_binary_target(scheduled_concurrency)

with open("../data/concurrency/scheduled_concurrency_optimized.csv", "w") as f:
    scheduled_concurrency.to_csv(f, index=False)

/tmp/ipykernel_1011790/2221210852.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  time_index = pd.date_range(start=start_time, end=end_time, freq="H")


Processing airport group: A
Processing airport group: B
Processing airport group: C
Processing airport group: D
Processing airport group: E
Processing airport group: F
Processing airport group: G


In [18]:
def compute_airport_congestion_dataset(scheduled_data_october):
    """For each timestep(hour or 30 minutes) from first entry in schedule to last, compute the number of scheduled departures and arrivals at each airport group."""
    # Determine the time range for the congestion dataset
    start_time = scheduled_data_october["std"].min().floor("h")
    end_time = scheduled_data_october["sta"].max().ceil("h")

    # Create a time index with hourly frequency
    time_index = pd.date_range(start=start_time, end=end_time, freq="h")

    # Get unique airports
    airports_groups = pd.unique(
        scheduled_data_october[["dep_airport", "arr_airport"]].values.ravel("K")
    )

    # Initialize an empty list to hold congestion records
    congestion_records = []

    # Iterate over each time step and airport to compute congestion
    for current_time in time_index:
        next_time = current_time + pd.Timedelta(hours=1)
        for airport_group in airports_groups:
            departures = (
                (scheduled_data_october["dep_airport"] == airport_group)
                & (scheduled_data_october["std"] >= current_time)
                & (scheduled_data_october["std"] < next_time)
            ).sum()
            arrivals = (
                (scheduled_data_october["arr_airport"] == airport_group)
                & (scheduled_data_october["sta"] >= current_time)
                & (scheduled_data_october["sta"] < next_time)
            ).sum()
            congestion_records.append(
                {
                    "timestamp_start": current_time,
                    "timestamp_end": next_time,
                    "airport": airport_group,
                    "departures": departures,
                    "arrivals": arrivals,
                }
            )

    # Convert the list of records to a DataFrame
    congestion_df = pd.DataFrame(congestion_records)
    return congestion_df

In [6]:
# print("Computing airport congestion dataset...")
# congestion_df = compute_airport_congestion_dataset(df_schedule)

Computing airport congestion dataset...


KeyboardInterrupt: 